In [ ]:
# ============================================================
# ASSIGNMENT 3 — QUESTION 2
# Degree Correlations: knn vs k Plot
# Dataset: Facebook Social Circles (SNAP)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import gzip
import os
import random
from collections import defaultdict

# seed for reproducible randomness
random.seed(42)
np.random.seed(42)
print("Libraries loaded.")

# ============================================================
# DOWNLOAD & LOAD THE DATASET
# ============================================================
data_url  = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
data_file = "facebook_combined.txt.gz"
txt_file  = "facebook_combined.txt"

if not os.path.exists(txt_file):
    print("Downloading dataset...")
    urllib.request.urlretrieve(data_url, data_file)
    with gzip.open(data_file, 'rb') as f_in:
        with open(txt_file, 'wb') as f_out:
            f_out.write(f_in.read())
    print("Download complete.")
else:
    print("Dataset already present.")

def load_graph(filepath):
    """load the graph from an edge list file"""
    adj   = defaultdict(set)
    edges = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            u, v = map(int, line.split())
            if u == v:
                continue
            if v not in adj[u]:
                adj[u].add(v)
                adj[v].add(u)
                edges.append((min(u,v), max(u,v)))
    nodes = sorted(adj.keys())
    return dict(adj), nodes, edges

adj_real, nodes_real, edges_real = load_graph(txt_file)
degrees_real = {n: len(adj_real[n]) for n in nodes_real}

print(f"Nodes  : {len(nodes_real)}")
print(f"Edges  : {len(edges_real)}")
print(f"Avg degree: {np.mean(list(degrees_real.values())):.4f}")

# ============================================================
# COMPUTE knn(k)
# ============================================================
def calc_knn(adj, nodes):
    """
    For each node compute average neighbour degree,
    then average over all nodes of the same degree k.
    Returns dict {k -> mean_knn(k)}
    """
    degree = {n: len(adj[n]) for n in nodes}

    knn_sum   = defaultdict(float)
    knn_count = defaultdict(int)

    for node in nodes:
        k = degree[node]
        if k == 0:
            continue
        knn_i = sum(degree[nb] for nb in adj[node]) / k
        knn_sum[k]   += knn_i
        knn_count[k] += 1

    return {k: knn_sum[k] / knn_count[k] for k in knn_sum}

knn_real = calc_knn(adj_real, nodes_real)
print(f"knn(k) computed for {len(knn_real)} distinct degree values.")

# ============================================================
# EDGE-SWAP RANDOMISATION
# ============================================================
def scramble_edges(adj_orig, edges_orig, nodes_orig, swap_factor=10):
    """
    Degree-preserving rewiring via edge swaps.
    Performs swap_factor * |E| successful swaps.
    Returns new adjacency dict and edge list.
    """
    adj   = {n: set(nb) for n, nb in adj_orig.items()}
    edges = list(edges_orig)
    n_edges   = len(edges)
    target    = swap_factor * n_edges
    success   = 0
    attempts  = 0
    max_tries = 100 * target

    while success < target and attempts < max_tries:
        attempts += 1
        i, j = random.sample(range(n_edges), 2)
        u, v = edges[i]
        s, t = edges[j]

        # skip if nodes share an endpoint
        if len({u, v, s, t}) < 4:
            continue
        if t in adj[u] or v in adj[s]:
            continue

        # ditch old edges, add new ones
        adj[u].discard(v); adj[v].discard(u)
        adj[s].discard(t); adj[t].discard(s)
        adj[u].add(t); adj[t].add(u)
        adj[s].add(v); adj[v].add(s)

        edges[i] = (min(u,t), max(u,t))
        edges[j] = (min(s,v), max(s,v))
        success += 1

    return adj, edges

# sanity check - make sure degrees didn't change
adj_test, edges_test = scramble_edges(adj_real, edges_real, nodes_real, swap_factor=10)
assert sorted([len(adj_real[n]) for n in nodes_real]) == \
       sorted([len(adj_test[n]) for n in nodes_real]), "Degree sequence mismatch!"
print("Edge-swap sanity check passed ✓")

# ============================================================
# AVERAGE knn(k) OVER A BUNCH OF RANDOM GRAPHS
# ============================================================
num_instances = 100
knn_accum = defaultdict(list)

print(f"Generating {num_instances} random graphs...")
for i in range(num_instances):
    adj_r, edges_r = scramble_edges(adj_real, edges_real, nodes_real, swap_factor=10)
    knn_r = calc_knn(adj_r, nodes_real)
    for k, val in knn_r.items():
        knn_accum[k].append(val)
    if (i+1) % 20 == 0:
        print(f"  {i+1}/{num_instances} done")

knn_rand_mean = {k: np.mean(v) for k, v in knn_accum.items()}
knn_rand_std  = {k: np.std(v)  for k, v in knn_accum.items()}
print("Done.")

# ============================================================
# PEARSON ASSORTATIVITY
# ============================================================
def calc_pearson_r(edges, degrees):
    """compute pearson assortativity coefficient"""
    M  = len(edges)
    if M == 0:
        return 0.0
    ki = np.array([degrees[u] for u,v in edges], dtype=float)
    kj = np.array([degrees[v] for u,v in edges], dtype=float)

    num = (np.sum(ki*kj)/M) - (np.sum(ki+kj)/(2*M))**2
    den = (np.sum(ki**2+kj**2)/(2*M)) - (np.sum(ki+kj)/(2*M))**2
    return 0.0 if den == 0 else num/den

r_real = calc_pearson_r(edges_real, degrees_real)
print(f"Assortativity (Real Network)     : r = {r_real:.4f}")

r_rand_vals = []
for _ in range(100):
    adj_r, edges_r = scramble_edges(adj_real, edges_real, nodes_real, swap_factor=10)
    deg_r = {n: len(adj_r[n]) for n in nodes_real}
    r_rand_vals.append(calc_pearson_r(edges_r, deg_r))

print(f"Assortativity (Random, avg/100)  : r = {np.mean(r_rand_vals):.4f} ± {np.std(r_rand_vals):.4f}")

# ============================================================
# PLOT IT
# ============================================================
# Real network data
k_real_sorted  = sorted(knn_real.keys())
knn_real_vals  = [knn_real[k] for k in k_real_sorted]

# Random network data (degrees with >= 5 samples for stability)
k_rand_sorted  = sorted([k for k,v in knn_accum.items() if len(v) >= 5])
knn_rand_m     = np.array([knn_rand_mean[k] for k in k_rand_sorted])
knn_rand_s     = np.array([knn_rand_std[k]  for k in k_rand_sorted])
k_rand_arr     = np.array(k_rand_sorted)

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(k_real_sorted, knn_real_vals, s=18, alpha=0.6,
           color='steelblue', label='Real Network (Facebook)', zorder=3)
ax.plot(k_real_sorted, knn_real_vals, color='steelblue', alpha=0.25, linewidth=1)

ax.plot(k_rand_arr, knn_rand_m, color='tomato', linewidth=2,
        label=f'Random Graphs (avg over {num_instances} instances)', zorder=3)
ax.fill_between(k_rand_arr, knn_rand_m - knn_rand_s, knn_rand_m + knn_rand_s,
                color='tomato', alpha=0.2, label='±1 std (random)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Node Degree $k$', fontsize=13)
ax.set_ylabel('Average Nearest-Neighbour Degree $k_{nn}(k)$', fontsize=13)
ax.set_title('Degree Correlations: $k_{nn}$ vs $k$\n'
             'Facebook Network vs Degree-Preserving Random Graphs', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.savefig('Q2_knn_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as Q2_knn_vs_k.png")
